# TFT — Walmart Store Sales Forecasting (with exogenous features)

**Architecture:** Temporal Fusion Transformer. Unlike DLinear, N-BEATS and PatchTST
(all univariate in this library), TFT can consume *exogenous* features via variable
selection networks + attention. This notebook tests the key question: **do the extra
features help, or is past sales enough?**

Two runs are compared:
- **Univariate** — only past sales `y`. Result: WMAE 2,268.
- **With exogenous** — plus historical (temperature, fuel, CPI, unemployment,
  markdowns) and static (store type, size) features. Result: WMAE 2,300.

**Finding:** the features did *not* help here — univariate was slightly better. Past
sales already carries the signal; the extra features added noise (also trained on a
small CPU budget). So the best TFT is the univariate one.

**Data view:** long format (`unique_id`, `ds`, `y`, exogenous), from
`src/features/nn_preprocessing.py`.
**Evaluation:** WMAE on the validation period (holiday weeks weighted 5x).
**MLflow structure:** experiment `TFT_Training` with runs `TFT_Data_Prep`,
`TFT_Univariate`, `TFT_With_Exogenous`, `TFT_Best_Pipeline`.

In [ ]:
# Run from the project root so data/ and src/ resolve, whether launched
# from the repo root or from the notebooks/ folder.
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

In [ ]:
import warnings
import numpy as np
import pandas as pd
import mlflow
import dagshub

from neuralforecast import NeuralForecast
from neuralforecast.models import TFT
from neuralforecast.losses.pytorch import MAE

from src.features.nn_preprocessing import (
    build_long_df, attach_static, temporal_split, FREQ, HIST_EXOG,
)
from src.evaluation.metrics import calculate_wmae

warnings.filterwarnings("ignore")

# Static (constant per store) features used as exogenous.
STAT_EXOG = ["Type_code", "Size"]

# Experiment tracking: DagsHub-hosted MLflow.
dagshub.init(repo_owner="GigiSichinava", repo_name="Walmart-Sales-Forecasting", mlflow=True)
mlflow.set_experiment("TFT_Training")

SPLIT_DATE = "2012-01-01"
SEED = 42
print("Tracking URI:", mlflow.get_tracking_uri())

In [ ]:
train_raw = pd.read_csv("data/train.csv")
stores = pd.read_csv("data/stores.csv")
features = pd.read_csv("data/features.csv")
print("train:", train_raw.shape, "| stores:", stores.shape, "| features:", features.shape)

## Run 1 — Data preparation

Build the long panel plus the static-feature table (`unique_id`, `Type_code`, `Size`).

In [ ]:
with mlflow.start_run(run_name="TFT_Data_Prep"):
    # Step 1: build the long panel, static table, and split by date.
    Y_df = build_long_df(train_raw, stores, features, fill_gaps=True)
    static_df = attach_static(Y_df, stores)
    train_df, valid_df, horizon = temporal_split(Y_df, SPLIT_DATE)

    # Step 2: log the preprocessing choices.
    mlflow.log_param("freq", FREQ)
    mlflow.log_param("split_date", SPLIT_DATE)
    mlflow.log_param("hist_exog", str(HIST_EXOG))
    mlflow.log_param("stat_exog", str(STAT_EXOG))
    mlflow.log_param("n_series", int(Y_df["unique_id"].nunique()))
    mlflow.log_param("horizon_weeks", int(horizon))

    print("series:", Y_df["unique_id"].nunique(), "| horizon (weeks):", horizon)

## WMAE evaluation helper

In [ ]:
def evaluate_forecast(forecast_df, valid_df, model_col="TFT"):
    # Join forecasts to the true validation values on series id and date.
    merged = forecast_df.merge(
        valid_df[["unique_id", "ds", "y", "IsHoliday"]],
        on=["unique_id", "ds"], how="inner",
    )
    # Neural models can output small negatives; sales can't be negative, so clip.
    preds = merged[model_col].clip(lower=0)
    wmae = calculate_wmae(merged["y"], preds, merged["IsHoliday"])
    return wmae, merged

## Run 2 — TFT univariate (no features)

Small TFT trained on past sales only, the reference for the feature test.
**Result: WMAE 2,268.**

In [ ]:
COMMON = dict(h=horizon, input_size=52, hidden_size=32, n_head=2,
              max_steps=200, scaler_type="robust", start_padding_enabled=True,
              random_seed=SEED, alias="TFT")

with mlflow.start_run(run_name="TFT_Univariate"):
    mlflow.log_params(COMMON)
    mlflow.log_param("exogenous", "none")

    nf = NeuralForecast(models=[TFT(loss=MAE(), **COMMON)], freq=FREQ)
    nf.fit(df=train_df[["unique_id", "ds", "y"]])

    fcst = nf.predict()
    wmae, _ = evaluate_forecast(fcst, valid_df, "TFT")
    mlflow.log_metric("WMAE", float(wmae))
    print(f"TFT univariate WMAE: {wmae:,.2f}")

## Run 3 — TFT with exogenous features

Same model, now fed the historical features (temperature, fuel, CPI, unemployment,
markdowns) and static features (store type, size). Historical exogenous need no future
values, so `predict()` works without a future dataframe. **Result: WMAE 2,300 — no
improvement over univariate.**

In [ ]:
with mlflow.start_run(run_name="TFT_With_Exogenous"):
    mlflow.log_params(COMMON)
    mlflow.log_param("exogenous", "hist+static")

    model = TFT(loss=MAE(), hist_exog_list=HIST_EXOG, stat_exog_list=STAT_EXOG, **COMMON)
    nf = NeuralForecast(models=[model], freq=FREQ)
    nf.fit(df=train_df[["unique_id", "ds", "y"] + HIST_EXOG], static_df=static_df)

    fcst = nf.predict()
    wmae, _ = evaluate_forecast(fcst, valid_df, "TFT")
    mlflow.log_metric("WMAE", float(wmae))
    print(f"TFT with-exogenous WMAE: {wmae:,.2f}")

## Run 4 — Best pipeline (retrain on all data, save for inference)

Univariate won (2,268 vs 2,300), so `USE_EXOG = False`. Retrain on the full history
and save for the inference step.

In [9]:
# Univariate had the lower WMAE, so features are not used in the saved model.
USE_EXOG = False

with mlflow.start_run(run_name="TFT_Best_Pipeline"):
    mlflow.log_params(COMMON)
    mlflow.log_param("exogenous", "hist+static" if USE_EXOG else "none")

    if USE_EXOG:
        model = TFT(loss=MAE(), hist_exog_list=HIST_EXOG, stat_exog_list=STAT_EXOG, **COMMON)
        nf_best = NeuralForecast(models=[model], freq=FREQ)
        nf_best.fit(df=Y_df[["unique_id", "ds", "y"] + HIST_EXOG], static_df=static_df)
    else:
        nf_best = NeuralForecast(models=[TFT(loss=MAE(), **COMMON)], freq=FREQ)
        nf_best.fit(df=Y_df[["unique_id", "ds", "y"]])

    save_path = "saved_models/tft_nf"
    os.makedirs(save_path, exist_ok=True)
    nf_best.save(path=save_path, overwrite=True, save_dataset=False)
    mlflow.log_artifacts(save_path, artifact_path="tft_nf")
    print("Saved TFT model to", save_path)

Seed set to 42
GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name                    | Type                     | Params | Mode 
-----------------------------------------------------------------------------
0 | loss                    | MAE                      | 0      | train
1 | padder_train            | ConstantPad1d            | 0      | train
2 | scaler                  | TemporalNorm             | 0      | train
3 | embedding               | TFTEmbedding             | 128    | train
4 | temporal_encoder        | TemporalCovariateEncoder | 39.6 K | train
5 | temporal_fusion_decoder | TemporalFusionDecoder    | 17.0 K | train
6 | output_adapter          | Linear                   | 33     | train
-----------------------------------------------------------------------------
56.8 K    Trainable params
0         Non-trainable params
56.8 K    Total params
0.227     Total estimated model params size (MB)
88        Modules in train mode
0         Modul

Epoch 0:  95%|█████████▌| 100/105 [01:43<00:05,  0.96it/s, v_num=28, train_loss_step=30.10]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|██████████| 95/95 [01:44<00:00,  0.91it/s, v_num=28, train_loss_step=22.10, train_loss_epoch=67.50]  
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|██████████| 95/95 [01:45<00:00,  0.90it/s, v_num=28, train_loss_step=22.10, train_loss_epoch=88.20]

`Trainer.fit` stopped: `max_steps=200` reached.


Epoch 1: 100%|██████████| 95/95 [01:45<00:00,  0.90it/s, v_num=28, train_loss_step=22.10, train_loss_epoch=88.20]
Saved TFT model to saved_models/tft_nf
🏃 View run TFT_Best_Pipeline at: https://dagshub.com/GigiSichinava/Walmart-Sales-Forecasting.mlflow/#/experiments/3/runs/9538cf1151884acb9b47bb5c567f8778
🧪 View experiment at: https://dagshub.com/GigiSichinava/Walmart-Sales-Forecasting.mlflow/#/experiments/3


## Notes

- TFT is the only model here that uses exogenous features.
- Univariate (2,268) slightly beat with-exogenous (2,300): features did not help here.
- TFT is close to N-BEATS (2,193), but N-BEATS is still the best neural model.